# AgriNav -- WeedDet Detector Training (Colab / GPU)

**Exploratory Phase-2 run.** This notebook only *drives* `agrinav.training.weeddet_train` from the repo -- it defines no model, loss, or training logic of its own (notebooks are disposable views; the code is the source of truth).

The goal of this first run is **loss convergence + qualitative validation predictions**, NOT a mAP number. There is deliberately **no COCO evaluator** here yet.

The **test split is sealed.** This notebook only ever touches `train.coco.json` (training) and `val.coco.json` (qualitative inspection). It never loads `test.coco.json`.

**Before you run:** `Runtime -> Change runtime type -> GPU`. You need, in Google Drive:
- `MyDrive/agrinav_data/detector_v1/split_v1/{train,val}.coco.json` -- run the **Detector Data Prep** notebook (`notebooks/detector_data_prep_colab.ipynb`) first if these are missing.
- `MyDrive/agrinav_data/RiceSEG.zip` -- the images referenced by each split record's archive-relative `file_name`.


## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code (clone the private repo)

**This repository is PRIVATE**, so an anonymous clone fails with
`could not read Username for 'https://github.com'` -- Colab has no interactive prompt.

Pick ONE of:

1. **Colab secret (recommended).** Left sidebar -> Secrets -> add `GITHUB_TOKEN` holding a
   fine-grained GitHub PAT with **Contents: Read** on this repo, and toggle notebook access on.
   The token is never printed, never saved into the notebook, and is stripped from the git
   remote after cloning.
2. **Make the repo public** -- then this cell works with no token at all.
3. **Skip GitHub:** upload the repo folder to Drive and set `REPO_DIR` to it.

The cell **stops immediately** if the code isn't available, so later cells can't cascade.


In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
BRANCH   = 'master'   # detector trainer merged to master 2026-07-23
REPO_DIR = '/content/agrinav'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

# GIT_TERMINAL_PROMPT=0 turns an auth failure into an immediate error, not a hang.
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED -- stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    # Never persist the token in .git/config
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], env=env)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], env=env)

os.chdir(REPO_DIR)
assert os.path.exists('src/agrinav/training/weeddet_train.py'), (
    f'Repo at {REPO_DIR} has no src/agrinav/training/weeddet_train.py -- wrong branch or upload?')
print('repo ready at', REPO_DIR)
get_ipython().system('git log --oneline -1')

## 4. Install the package

Colab already ships a CUDA build of torch/torchvision that satisfies the `train` extra's
version range, so pip keeps it and only adds the remaining deps. The editable install is what
makes `python -m agrinav.training.weeddet_train` importable.


In [ ]:
get_ipython().system('pip install -q -e ".[train]"')

## 5. Locate the split + extract images

Verifies the train/val split JSONs are in Drive (the **test** split is never referenced), then
extracts `RiceSEG.zip` to fast local Colab disk at `/content/riceseg`. Each split record's
`file_name` is archive-relative (e.g. `global rice segmentation/China/GD/rgb/...jpg`), so the
detector resolves images as `os.path.join(images_root, file_name)` with `images_root=/content/riceseg`.


In [ ]:
import os, glob, zipfile

DET        = '/content/drive/MyDrive/agrinav_data/detector_v1/split_v1'
TRAIN_JSON = f'{DET}/train.coco.json'
VAL_JSON   = f'{DET}/val.coco.json'
for f in (TRAIN_JSON, VAL_JSON):
    assert os.path.exists(f), (
        f'{f} missing -- run the Detector Data Prep notebook first '
        '(notebooks/detector_data_prep_colab.ipynb).')
# NOTE: we never open test.coco.json. The test split stays sealed until a final locked eval.

IMAGES_ROOT = '/content/riceseg'
ricezip = '/content/drive/MyDrive/agrinav_data/RiceSEG.zip'
if not os.path.exists(ricezip):
    hits = glob.glob('/content/drive/MyDrive/**/RiceSEG.zip', recursive=True)
    assert hits, 'RiceSEG.zip not found in Drive. Upload it (mirrors the pretraining notebook).'
    ricezip = hits[0]
print('RiceSEG.zip:', ricezip)

os.makedirs(IMAGES_ROOT, exist_ok=True)
if not os.path.isdir(os.path.join(IMAGES_ROOT, 'global rice segmentation')):
    with zipfile.ZipFile(ricezip) as z:
        z.extractall(IMAGES_ROOT)
print('images_root:', IMAGES_ROOT)
print('sample countries:', os.listdir(os.path.join(IMAGES_ROOT, 'global rice segmentation'))[:6])

## 6. Sanity gate: self-test

`--self-test` runs one forward+backward on synthetic tensors (including a zero-GT image) with
no data and no network. It must print PASS before spending GPU time on the full run.


In [ ]:
# -u forces unbuffered stdout so the line streams live in Colab.
get_ipython().system('python -B -u -m agrinav.training.weeddet_train --self-test')

## 7. Train (exploratory)

Uses `configs/training/detector_gpu.yaml` (512px, batch 8, ImageNet warm-start, AMP, ~18 epochs).
Writes checkpoints to a timestamped **Drive** run dir plus an immutable `run_manifest.json`
(git commit, config, class map, seed, split files). `weeddet_best.pth` is selected by lowest
**train** loss -- this is an exploratory convergence run, not a validated model.


In [ ]:
import json, subprocess, datetime

TS      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'/content/drive/MyDrive/agrinav_data/detector_v1/runs/weeddet_{TS}'
os.makedirs(RUN_DIR, exist_ok=True)

CLASS_NAMES = 'rice_protect,weed_target,non_target_aquatic'
SEED        = 42
CONFIG      = 'configs/training/detector_gpu.yaml'

def _git(*a):
    try:
        return subprocess.check_output(['git', '-C', '.', *a], text=True).strip()
    except Exception:
        return None

manifest = {
    'run_id': f'weeddet_{TS}',
    'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'git_commit': _git('rev-parse', 'HEAD'),
    'git_branch': _git('rev-parse', '--abbrev-ref', 'HEAD'),
    'config_file': CONFIG,
    'class_map': {n: i for i, n in enumerate(CLASS_NAMES.split(','))},
    'seed': SEED,
    'train_split': TRAIN_JSON,
    'val_split': VAL_JSON,
    'split_manifest': (f'{DET}/split-v1.json' if os.path.exists(f'{DET}/split-v1.json') else None),
    'images_root': IMAGES_ROOT,
    'note': ('Exploratory run: loss convergence + qualitative val predictions only, '
             'no mAP. Test split sealed.'),
}
with open(f'{RUN_DIR}/run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
print('run dir :', RUN_DIR)
print(json.dumps(manifest, indent=2))

In [ ]:
# -u keeps per-epoch loss lines streaming live. --checkpoint-dir overrides the config default.
get_ipython().system(
    'python -B -u -m agrinav.training.weeddet_train'
    f' --ann-file "{TRAIN_JSON}"'
    f' --images-root "{IMAGES_ROOT}"'
    f' --class-names "{CLASS_NAMES}"'
    f' --config "{CONFIG}"'
    f' --seed {SEED}'
    f' --checkpoint-dir "{RUN_DIR}"')

## 8. Qualitative predictions on VALIDATION images

Loads the best (EMA) checkpoint and draws predicted boxes on ~6 **validation** images for visual
inspection -- this is the exploratory deliverable, **not** a mAP score. Each box is labelled with
its class **name** and score (never colour alone). The **test split is never touched.**


In [ ]:
import random, torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from agrinav.training.weeddet_train import (
    _CocoSplitDataset, load_checkpoint_model, predict_image)

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
class_names = CLASS_NAMES.split(',')
ckpt        = f'{RUN_DIR}/weeddet_best.pth'
assert os.path.exists(ckpt), f'no checkpoint at {ckpt} -- did training finish?'
model = load_checkpoint_model(ckpt, num_classes=len(class_names), device=device)

# VALIDATION images only -- the test split stays sealed.
val_ds = _CocoSplitDataset(VAL_JSON, IMAGES_ROOT, tuple(class_names), img_size=512)
items  = val_ds.items()
random.Random(0).shuffle(items)
picks  = items[:6]

colors = ['lime', 'red', 'cyan']
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, (img_id, path, W, H) in zip(axes.ravel(), picks):
    img, boxes, scores, labels = predict_image(
        model, path, img_size=512, device=device, score_thr=0.3)
    ax.imshow(img); ax.set_axis_off()
    ax.set_title(f'{os.path.basename(path)}  ({len(boxes)} dets @0.3)', fontsize=9)
    for (x1, y1, x2, y2), s, l in zip(boxes, scores, labels):
        c = colors[int(l) % len(colors)]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=c, linewidth=2))
        ax.text(x1, max(y1 - 4, 0), f'{class_names[int(l)]} {s:.2f}', color=c, fontsize=8,
                bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))
plt.suptitle('WeedDet exploratory predictions on VAL (no mAP yet; test split sealed)', fontsize=13)
plt.tight_layout(); plt.show()

## Next steps

1. Read the per-epoch `avg_loss` stream above -- the first exploratory goal is a **smoothly
   decreasing training loss** with no NaNs, plus qualitatively sensible boxes on VAL.
2. Artifacts are in the Drive run dir: `weeddet_best.pth`, periodic `weeddet_epochN.pth`, and
   `run_manifest.json` (git commit + config + class map + seed + split files).
3. A calibrated operating point, a COCO mAP evaluator, and the **sealed** test-set evaluation are
   deliberately out of scope for this exploratory pass.
